# DroneFuse - T3: the recording check on TSMS-Drone and DroneRFc-MM (GPU T4, Internet On)

The same three questions as on MMAUD, asked on two independent datasets:
1. **Probes**: how well does each sensor tell the drone type apart, and how well does it tell two sessions of the
   same drone apart?
2. **Within-session vs cross-session accuracy**: training and test data from the same sessions (time blocks), or
   from different sessions (other distances in TSMS-Drone, the other flight in DroneRFc-MM).
3. **Unknown-type rejection** when the known test data come from sessions seen in training vs from new sessions.

Features are frozen (ConvNeXt-T for images, log-mel statistics for sound, statistics of the radar returns) and
every model is a logistic regression, so the comparison is about the data and the protocol, not about tuning.

**Kaggle settings:** Accelerator = **GPU T4**. Internet = **On** (downloads the ConvNeXt weights).
**Inputs to add:** `dronefuse-code` (latest), `dronefuse-tsms` (output of T1, all parts) and `dronefuse-rfcmm-core`
(output of T2). A dataset that is not attached is skipped.
**Run time:** about 1 h (most of it reading 112,500 TSMS images and 18 videos).
**Output:** `t3/t3_report.txt`, `t3/t3_results.json`, and the cached features (attach this output to a rerun to
skip the feature step).

In [ ]:
import os
import sys
from pathlib import Path

# The DroneFuse code is attached as a Kaggle dataset. Find the `dronefuse`
# package wherever Kaggle placed it, and make it importable.
def find_file(name, root='/kaggle/input'):
    # os.walk with followlinks=True also enters inputs that Kaggle mounts as links.
    for folder, _, files in os.walk(root, followlinks=True):
        if name in files:
            return Path(folder) / name
    return None


package = find_file('rosbag_index.py')
if package is None:
    attached = sorted(str(p) for p in Path('/kaggle/input').glob('*/*'))[:20]
    raise RuntimeError('The dronefuse-code dataset is not attached to this notebook. In the editor: '
                       'Add Input -> Your Datasets -> dronefuse-code, then Save & Run All again. '
                       f'Attached now: {attached or "nothing"}')
if (package.parent / '__init__.py').exists() and package.parent.name == 'dronefuse':
    code_root = package.parent.parent
else:
    # Kaggle sometimes flattens an uploaded folder, so the files sit directly in
    # the dataset with no `dronefuse/` folder around them. In that case, link the
    # folder that holds rosbag_index.py under the name `dronefuse`.
    flat = package
    code_root = Path('/kaggle/working/code')
    code_root.mkdir(parents=True, exist_ok=True)
    if not (code_root / 'dronefuse').exists():
        os.symlink(flat.parent, code_root / 'dronefuse')

sys.path.insert(0, str(code_root))
import dronefuse
print('DroneFuse code loaded from', Path(dronefuse.__file__).parent)

In [ ]:
# ---- Settings (the only cell you should need to edit) ----
INPUT = Path('/kaggle/input')
OUTPUT = Path('/kaggle/working/t3')
ALLOW_CPU = False            # True only to test without a GPU (the TSMS images then take hours)
N_BOOT = 1000
TIME_BUDGET_MINUTES = 11 * 60

In [ ]:
import json
import numpy as np
from dronefuse import environment, extra
from dronefuse.runner import StepRunner

OUTPUT.mkdir(parents=True, exist_ok=True)
runner = StepRunner(OUTPUT, TIME_BUDGET_MINUTES)
summary = {'environment': runner.run('measure the machine', environment.collect)}
summary['gpu'] = environment.check_gpu(ALLOW_CPU)
tsms_dirs = sorted({p.parent for p in INPUT.rglob('FMCW_*.tar')})
rfcmm_roots = sorted({p.parent.parent.parent for p in INPUT.rglob('PTZ_meta.yaml')})
caches = sorted({p.parent for p in INPUT.rglob('t3_cache_marker.txt')})
print('TSMS tar folders:', [str(p) for p in tsms_dirs] or 'none')
print('DroneRFc-MM root:', [str(p) for p in rfcmm_roots] or 'none')
print('earlier feature caches:', [str(p) for p in caches] or 'none')
CACHE = OUTPUT / 'cache'
CACHE.mkdir(exist_ok=True)
(CACHE / 't3_cache_marker.txt').write_text('features of T3')
for old in caches:                       # reuse features of an earlier run
    for f in old.rglob('*'):
        target = CACHE / f.relative_to(old)
        if f.is_file() and not target.exists():
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(f.read_bytes())
embed = runner.run('load ConvNeXt-T', extra.make_embedder)
results = {}

def save():
    with open(OUTPUT / 't3_results.json', 'w') as f:
        json.dump({'summary': summary, 'results': results}, f, indent=1, default=str)

In [ ]:
# TSMS-Drone: embed every image once (cached per tar), join the three sensors, run the experiments.
if tsms_dirs and embed is not None:
    per = runner.run('TSMS features', extra.tsms_features, tsms_dirs, CACHE / 'tsms', embed)
    if per:
        tsms = extra.tsms_dataset(per)
        print('TSMS rows (simultaneous measurements):', tsms['rows'], 'images per sensor:', tsms['counts'])
        results['tsms'] = runner.run('TSMS experiments', extra.run_tsms, tsms, N_BOOT)
        save()
else:
    print('TSMS-Drone not attached: skipped')

In [ ]:
# DroneRFc-MM: 1 s windows over the time both cameras recorded; PTZ and fisheye frames, audio, radar.
import pickle
if rfcmm_roots and embed is not None:
    cache = CACHE / 'rfcmm_features.pkl'
    if cache.exists():
        rf = pickle.loads(cache.read_bytes())
    else:
        rf = runner.run('DroneRFc-MM features', extra.rfcmm_features, rfcmm_roots[0], embed)
        if rf:
            cache.write_bytes(pickle.dumps(rf))
    if rf:
        print('DroneRFc-MM windows with every sensor:', rf['rows'], 'missing per sensor:', rf['missing'])
        results['rfcmm'] = runner.run('DroneRFc-MM experiments', extra.run_rfcmm, rf, N_BOOT)
        save()
else:
    print('DroneRFc-MM not attached: skipped')

In [ ]:
# Report
def table(acc, methods):
    rows = []
    for m in methods:
        r = acc.get(m)
        if r:
            line = f"    {m:10s} accuracy {r['accuracy']:5.1f} [{r['ci'][0]:5.1f}, {r['ci'][1]:5.1f}]"
            if 'auroc_msp' in r:
                line += f"   unknown AUROC {r['auroc_msp']:5.1f} [{r['auroc_ci'][0]:5.1f}, {r['auroc_ci'][1]:5.1f}]"
            rows.append(line)
    return rows

lines = [f"MACHINE  GPU {summary.get('gpu')}"]
for name, res in results.items():
    if not res:
        lines.append(f"\n{name}: FAILED (see the step log)")
        continue
    sensors = sorted(res['probes'])
    methods = sensors + ['concat', 'late']
    lines.append(f"\n==================== {name.upper()} ====================")
    lines.append(f"rows {res['rows']}  types {res['types']}")
    if 'sessions' in res:
        lines.append(f"sessions {res['sessions']}  missing per sensor {res['missing']}")
    lines.append('PROBES (train -> val, balanced accuracy %): type | session')
    for s in sensors:
        p = res['probes'][s]
        lines.append(f"    {s:10s} type {p['type']} ({p['type_classes']} classes) | session {p['session']} ({p['session_classes']} classes)")
    for prot, acc in res['accuracy'].items():
        w = acc['_windows']
        lines.append(f"{prot.upper()}  (train {w['train']}, test known {w['test_known']}, unknown {w['test_unknown']})")
        lines += table(acc, methods)
    lines.append('LEAVE-ONE-TYPE-OUT unknown AUROC (MSP, %)')
    for held, r in res['leave_one_out'].items():
        if 'within' in r:
            lines.append(f"    {held:18s} within-session: " + ', '.join(f"{m} {r['within'].get(m)}" for m in methods) +
                         f"\n    {'':18s} cross-distance: " + ', '.join(f"{m} {r['cross'].get(m)}" for m in methods))
        else:
            lines.append(f"    {held:18s} " + ', '.join(f"{m} {r.get(m)}" for m in methods))
failed = runner.failed()
lines.append('\nSTEPS: ' + ('all succeeded' if not failed else 'not succeeded: ' + ', '.join(failed)))
report = '\n'.join(lines)
(OUTPUT / 't3_report.txt').write_text(report)
save()
print(report)

## After the run

Send back `t3/t3_report.txt` (and `t3/t3_results.json` if possible). If a step failed, save the output as a dataset,
attach it, and run again: the cached features are reused.